# DINOv3 ViT-B/16 — linear probe on CIFAR-10

Freeze the backbone, extract CLS features once (cached), train a single `nn.Linear` head. Should reach ~95%+ test accuracy in a couple of minutes on a modest GPU.

Swap the dataset block (cell 3) for `datasets.ImageFolder('path/to/your/data')` to use your own labeled data.

In [1]:
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
WEIGHTS = PROJECT_DIR / 'weights' / 'dinov3_vitb16_pretrain_lvd1689m.pth'
DATA_DIR = PROJECT_DIR / 'data'
CACHE = DATA_DIR / 'dinov3_cifar10_features.pt'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device, WEIGHTS.exists()

('cuda', True)

In [2]:
backbone = torch.hub.load('facebookresearch/dinov3', 'dinov3_vitb16', pretrained=False)
backbone.load_state_dict(torch.load(WEIGHTS, map_location='cpu', weights_only=True))
for p in backbone.parameters():
    p.requires_grad = False
backbone.eval().to(device);

Using cache found in /home/henrytan/.cache/torch/hub/facebookresearch_dinov3_main


In [3]:
from datasets import load_dataset                                                                                                  
                                   
class HFWrap(torch.utils.data.Dataset):                                                                                            
  def __init__(self, hf_ds, transform):
      self.ds, self.transform = hf_ds, transform                                                                                 
  def __len__(self): return len(self.ds)        
  def __getitem__(self, i):                                                                                                      
      row = self.ds[i]     
      return self.transform(row['img']), row['label']                                                                            
                                                     
transform = transforms.Compose([                                                                                                   
  transforms.Resize(224),     
  transforms.CenterCrop(224),                                                                                                    
  transforms.ToTensor(),     
  transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),                                                            
])                                                                     
                                                                                                                                 
train_hf = load_dataset("uoft-cs/cifar10", split="train")
test_hf  = load_dataset("uoft-cs/cifar10", split="test")                                                                           
train_set = HFWrap(train_hf, transform)                 
test_set  = HFWrap(test_hf, transform)                                                                                             
len(train_set), len(test_set), train_hf.features['label'].names  

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

(50000,
 10000,
 ['airplane',
  'automobile',
  'bird',
  'cat',
  'deer',
  'dog',
  'frog',
  'horse',
  'ship',
  'truck'])

In [4]:
@torch.no_grad()
def extract(dataset, batch_size=64, num_workers=4):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    feats, labels = [], []
    for x, y in loader:
        feats.append(backbone(x.to(device)).cpu())
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)

if CACHE.exists():
    cached = torch.load(CACHE, weights_only=True)
    Xtr, ytr, Xte, yte = cached['Xtr'], cached['ytr'], cached['Xte'], cached['yte']
    print('loaded cached features')
else:
    print('extracting features (one-time, a few minutes)...')
    Xtr, ytr = extract(train_set)
    Xte, yte = extract(test_set)
    CACHE.parent.mkdir(exist_ok=True)
    torch.save({'Xtr': Xtr, 'ytr': ytr, 'Xte': Xte, 'yte': yte}, CACHE)

Xtr.shape, Xte.shape

extracting features (one-time, a few minutes)...


(torch.Size([50000, 768]), torch.Size([10000, 768]))

In [5]:
num_classes = int(ytr.max()) + 1
head = nn.Linear(Xtr.shape[1], num_classes).to(device)
opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

Xtr_d, ytr_d = Xtr.to(device), ytr.to(device)
Xte_d, yte_d = Xte.to(device), yte.to(device)

batch_size = 1024
for epoch in range(20):
    perm = torch.randperm(len(Xtr_d))
    head.train()
    losses = []
    for i in range(0, len(Xtr_d), batch_size):
        idx = perm[i:i + batch_size]
        logits = head(Xtr_d[idx])
        loss = loss_fn(logits, ytr_d[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    head.eval()
    with torch.no_grad():
        acc = (head(Xte_d).argmax(1) == yte_d).float().mean().item()
    print(f'epoch {epoch:2d}  loss {sum(losses)/len(losses):.4f}  test acc {acc*100:.2f}%')

epoch  0  loss 0.8082  test acc 96.71%
epoch  1  loss 0.1760  test acc 97.32%
epoch  2  loss 0.1220  test acc 97.57%
epoch  3  loss 0.0997  test acc 97.83%
epoch  4  loss 0.0869  test acc 98.03%
epoch  5  loss 0.0784  test acc 98.06%
epoch  6  loss 0.0723  test acc 98.08%
epoch  7  loss 0.0676  test acc 98.09%
epoch  8  loss 0.0639  test acc 98.11%
epoch  9  loss 0.0607  test acc 98.11%
epoch 10  loss 0.0582  test acc 98.14%
epoch 11  loss 0.0561  test acc 98.16%
epoch 12  loss 0.0540  test acc 98.17%
epoch 13  loss 0.0522  test acc 98.18%
epoch 14  loss 0.0507  test acc 98.16%
epoch 15  loss 0.0493  test acc 98.20%
epoch 16  loss 0.0479  test acc 98.16%
epoch 17  loss 0.0467  test acc 98.22%
epoch 18  loss 0.0456  test acc 98.20%
epoch 19  loss 0.0445  test acc 98.22%


## Bonus: k-NN classifier (zero training)

Often within 1–2% of the linear probe — useful as a quick baseline before training anything.

In [6]:
import torch.nn.functional as F

Xtr_n = F.normalize(Xtr_d, dim=1)
Xte_n = F.normalize(Xte_d, dim=1)

k = 20
sims = Xte_n @ Xtr_n.T                       # (n_test, n_train)
topk_sim, topk_idx = sims.topk(k, dim=1)
topk_labels = ytr_d[topk_idx]                # (n_test, k)

votes = torch.zeros(len(Xte_d), num_classes, device=device)
votes.scatter_add_(1, topk_labels, topk_sim)
knn_acc = (votes.argmax(1) == yte_d).float().mean().item()
print(f'k-NN (k={k}) test acc: {knn_acc*100:.2f}%')

k-NN (k=20) test acc: 97.97%
